In [21]:
import jams
import numpy as np
import os
import pretty_midi
from music21 import stream, note, tempo, meter, clef, articulations, expressions, spanner, interval, instrument
from music21 import note as m21_note  # Ensure we have access to Rest
import random
np.int = int # deprecated np.int
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple
from collections import defaultdict
import math

In [22]:
midi_dir= '/data/akshaj/MusicAI/GuitarSet/MIDIAnnotations/'
midi_path = os.path.join(midi_dir, '05_Funk2-108-Eb_solo.mid')

In [23]:
# Standard guitar tuning (MIDI pitch for each open string)
STANDARD_TUNING = {
    6: 40,  # E2 (low E)
    5: 45,  # A2
    4: 50,  # D3
    3: 55,  # G3
    2: 59,  # B3
    1: 64   # E4 (high e)
}

In [24]:
def midi_pitch_to_guitar_positions(midi_pitch, tuning=STANDARD_TUNING, max_fret=12):
    """
    Find all possible (string, fret) positions for a given MIDI pitch.
    
    Args:
        midi_pitch: MIDI note number (e.g., 60 = middle C)
        tuning: Dictionary of {string_number: open_string_midi_pitch}
        max_fret: Maximum fret to consider
    
    Returns:
        List of (string, fret) tuples, sorted by preference
    """
    positions = []
    
    for string_num in range(1, 7):  # Strings 1-6
        open_pitch = tuning[string_num]
        fret = midi_pitch - open_pitch
        
        # Valid if fret is 0-12 (or your max_fret)
        if 0 <= fret <= max_fret:
            positions.append((string_num, fret))
    
    # Sort by preference: middle strings first, lower frets preferred
    # This creates more natural fingering
    def position_score(pos):
        string, fret = pos
        # Prefer middle strings (3, 4) and lower frets
        string_penalty = abs(string - 3.5)  # Prefer strings 3-4
        fret_penalty = fret * 0.1  # Slight preference for lower frets
        return string_penalty + fret_penalty
    
    positions.sort(key=position_score)
    return positions

In [25]:
def choose_best_position(midi_pitch, previous_position=None, tuning=STANDARD_TUNING):
    """
    Choose the best string/fret position for a pitch.
    Takes into account the previous position to minimize hand movement.
    
    Args:
        midi_pitch: MIDI note number
        previous_position: Previous (string, fret) tuple, or None
        tuning: Guitar tuning
    
    Returns:
        (string, fret) tuple
    """
    positions = midi_pitch_to_guitar_positions(midi_pitch, tuning)
    
    if not positions:
        # Pitch out of range - use closest approximation
        # Find the closest string
        closest_string = min(tuning.keys(), 
                           key=lambda s: abs(tuning[s] - midi_pitch))
        fret = max(0, min(12, midi_pitch - tuning[closest_string]))
        return (closest_string, fret)
    
    if previous_position is None:
        # No previous position - use most natural position
        return positions[0]
    
    # Choose position closest to previous position (minimize hand movement)
    prev_string, prev_fret = previous_position
    
    def distance_score(pos):
        string, fret = pos
        string_distance = abs(string - prev_string)
        fret_distance = abs(fret - prev_fret)
        # Weight fret distance more (moving along frets is harder than changing strings)
        return fret_distance * 2 + string_distance
    
    return min(positions, key=distance_score)

In [26]:
def midi_to_jams_with_tablature(midi_path, tuning=STANDARD_TUNING):
    """
    Convert MIDI file to JAMS with intelligent tablature mapping.
    Extracts tempo and time signature directly from the MIDI file.
    
    Args:
        midi_path: Path to MIDI file
        tuning: Guitar tuning dictionary
    
    Returns:
        JAMS object with both note_midi and tab_note annotations,
        plus extracted tempo_bpm and time_signature
    """
    # Load MIDI
    pm = pretty_midi.PrettyMIDI(midi_path)
    guitar_notes = pm.instruments[0].notes
    
    # Extract tempo from MIDI
    # get_tempo_changes() returns (times, tempos) arrays
    tempo_times, tempos = pm.get_tempo_changes()
    if len(tempos) > 0:
        tempo_bpm = tempos[0]  # Use first tempo (or could average/use most common)
    else:
        tempo_bpm = 120  # Default fallback
    
    # Extract time signature from MIDI
    # time_signature_changes is a list of TimeSignature objects
    if pm.time_signature_changes:
        ts = pm.time_signature_changes[0]  # Use first time signature
        time_sig_numerator = ts.numerator
        time_sig_denominator = ts.denominator
        print(f"Time signature from MIDI: {time_sig_numerator}/{time_sig_denominator}")
    else:
        print("No time signature found, using default 4/4 instead")
        time_sig_numerator = 4  # Default 4/4
        time_sig_denominator = 4
    
    time_signature = f"{time_sig_numerator}/{time_sig_denominator}"
    
    print(f"Extracted from MIDI - Tempo: {tempo_bpm:.1f} BPM, Time Signature: {time_signature}")
    
    # Create JAMS
    jam = jams.JAMS()
    
    # Store tempo and time signature in JAMS file metadata
    jam.file_metadata.duration = pm.get_end_time()
    jam.sandbox.tempo_bpm = tempo_bpm
    jam.sandbox.time_signature = time_signature
    jam.sandbox.time_sig_numerator = time_sig_numerator
    jam.sandbox.time_sig_denominator = time_sig_denominator
    
    # Add note_midi annotation
    note_ann = jams.Annotation(namespace='note_midi')
    for n in guitar_notes:
        note_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=n.pitch,
            confidence=n.velocity / 127
        )
    jam.annotations.append(note_ann)
    
    # Add tab_note annotation with intelligent string/fret mapping
    tab_ann = jams.Annotation(namespace='tab_note')
    
    previous_position = None
    for n in guitar_notes:
        # Find best position for this pitch
        string, fret = choose_best_position(n.pitch, previous_position, tuning)
        previous_position = (string, fret)
        
        value = {
            "pitch": n.pitch,
            "string": string,
            "fret": fret,
            "techniques": []  # No techniques from MIDI
        }
        
        tab_ann.append(
            time=n.start,
            duration=n.end - n.start,
            value=value,
            confidence=n.velocity / 127
        )
    
    jam.annotations.append(tab_ann)
    
    print(f"Converted {len(guitar_notes)} notes to tablature")
    print(f"Pitch range: {min(n.pitch for n in guitar_notes)} - {max(n.pitch for n in guitar_notes)}")
    
    return jam

In [ ]:
def jams_to_musicxml_real(jam, output_xml='output.xml', tempo_bpm=120):
    """
    Convert JAMS with real tablature to MusicXML.
    Uses actual pitch information for correct notation.
    
    Args:
        jam: JAMS object with tab_note annotation
        output_xml: Output MusicXML file path
        tempo_bpm: Tempo in BPM
    
    Returns:
        Path to created XML file
    """
    
    # Find tab_note annotation
    tab_notes = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_notes = ann
            break
    
    if tab_notes is None:
        raise ValueError("No tab_note annotation found")
    
    print(f"Converting {len(tab_notes.data)} notes to MusicXML...")
    
    # Create score
    score = stream.Score()
    part = stream.Part()

    # Add instrument data
    guitar = instrument.Guitar()
    guitar.instrumentName = 'Classical Guitar (tablature)'
    guitar.instrumentAbbreviation = 'Guit.'
    guitar.instrumentSound = 'pluck.guitar.nylon-string'
    part.insert(0, guitar)
    
    # Add metadata
    part.insert(0, tempo.MetronomeMark(number=tempo_bpm))
    
    # Beat duration in seconds
    beat_sec = 60.0 / tempo_bpm
    # Setting correct time siganture
    time_sig_numerator = getattr(jam.sandbox, 'time_sig_numerator', 4)
    time_sig_denominator = getattr(jam.sandbox, 'time_sig_denominator', 4)
    # Calculate beats per measure
    # The numerator tells us how many beats, but we need to account for the denominator
    # In 4/4: 4 quarter-note beats per measure
    # In 6/8: 6 eighth-note beats = 3 quarter-note beats per measure
    # In 3/4: 3 quarter-note beats per measure
    beats_per_measure = time_sig_numerator * (4.0 / time_sig_denominator)
    ts = meter.TimeSignature(f'{time_sig_numerator}/{time_sig_denominator}')
    part.insert(0, ts)
    part.insert(0, clef.TabClef())
    
    # Sort notes by time and collect quantized note data
    note_events = []
    for obs in sorted(tab_notes.data, key=lambda x: x.time):
        val = obs.value
        
        # Quantize time position to nearest eighth note
        time_beats = obs.time / beat_sec
        quantized_time = round(time_beats * 2) / 2
        
        # Quantize duration to standard values
        duration_beats = obs.duration / beat_sec
        quantized_duration = round(duration_beats * 2) / 2
        if quantized_duration < 0.5:
            quantized_duration = 0.5  # Minimum eighth note
        
        note_events.append({
            'time': quantized_time,
            'duration': quantized_duration,
            'pitch': val['pitch'],
            'string': val['string'],
            'fret': val['fret'],
            'techniques': val.get('techniques', [])
        })
    
    # Calculate total duration and number of complete measures needed
    if note_events:
        last_note_end = max(n['time'] + n['duration'] for n in note_events)
    else:
        last_note_end = 0
    
    total_measures = math.ceil(last_note_end / beats_per_measure)
    if total_measures < 1:
        total_measures = 1
    total_beats = total_measures * beats_per_measure
    
    print(f"  Total duration: {last_note_end:.2f} beats -> {total_measures} measures")
    
    # Build a timeline of events (notes and rests) to fill each measure completely
    # We'll process measure by measure
    current_beat = 0.0
    previous_note = None
    
    for measure_num in range(total_measures):
        measure_start = measure_num * beats_per_measure
        measure_end = measure_start + beats_per_measure
        position_in_measure = 0.0  # Track our position within the current measure
        
        # Get notes that start in this measure
        measure_notes = [n for n in note_events if measure_start <= n['time'] < measure_end]
        measure_notes.sort(key=lambda x: x['time'])
        
        for note_event in measure_notes:
            note_start_in_measure = note_event['time'] - measure_start
            
            # If there's a gap before this note, insert rest(s)
            if note_start_in_measure > position_in_measure + 0.001:  # Small epsilon for float comparison
                rest_duration = note_start_in_measure - position_in_measure
                rest_start = measure_start + position_in_measure
                
                # Add rest
                r = m21_note.Rest()
                r.quarterLength = rest_duration
                part.insert(rest_start, r)
                position_in_measure = note_start_in_measure
            
            # Calculate how much of this note fits in this measure
            note_end_in_measure = note_start_in_measure + note_event['duration']
            
            # Truncate note if it extends beyond measure boundary
            if note_end_in_measure > beats_per_measure:
                actual_duration = beats_per_measure - note_start_in_measure
            else:
                actual_duration = note_event['duration']
            
            # Create note with correct pitch
            n = note.Note()
            n.pitch.midi = note_event['pitch']
            n.quarterLength = actual_duration
            
            # Add tablature info
            n.editorial.stringNumber = note_event['string']
            n.editorial.fretNumber = note_event['fret']
            
            # Apply techniques to note
            techniques = note_event['techniques']
            if techniques:
                print(f"  Processing {len(techniques)} technique(s)")
                for tech in techniques:
                    if not tech:
                        continue
                    
                    if tech in ['hammer-on', 'hammer_on']:
                        print("hammer-on detected")
                        n.expressions.append(expressions.TextExpression('H'))
                        if previous_note is not None:
                            slur = spanner.Slur(previous_note, n)
                            part.insert(0, slur)
                    
                    elif tech in ['pull-off', 'pull_off']:
                        print("pull off detected")
                        n.expressions.append(expressions.TextExpression('P'))
                        if previous_note is not None:
                            slur = spanner.Slur(previous_note, n)
                            part.insert(0, slur)
                    
                    elif tech == 'bend':
                        print("bend detected")
                        bend_amount = 0.5
                        bend = articulations.FretBend(
                            bendAlter=interval.Interval(bend_amount)
                        )
                        n.articulations.append(bend)
                        n.expressions.append(expressions.TextExpression('B'))
                    
                    elif tech == 'slide':
                        print("slide detected")
                        n.expressions.append(expressions.TextExpression('/'))
                    
                    elif tech == 'vibrato':
                        print("Vibrato detected")
                        n.expressions.append(expressions.TextExpression('~'))
                    
                    elif tech == 'harmonic':
                        print("Harmonic detected")
                        n.articulations.append(articulations.Harmonic())
                        harm_text = expressions.TextExpression('Harm.')
                        harm_text.placement = 'above'
                        harm_text.style.fontStyle = 'italic'
                        harm_text.style.fontSize = 10
                        part.insert(note_event['time'], harm_text)
            
            # Insert note at its absolute position
            part.insert(note_event['time'], n)
            previous_note = n
            
            position_in_measure = note_start_in_measure + actual_duration
        
        # Fill rest of measure with rest if needed
        if position_in_measure < beats_per_measure - 0.001:
            remaining = beats_per_measure - position_in_measure
            r = m21_note.Rest()
            r.quarterLength = remaining
            rest_position = measure_start + position_in_measure
            part.insert(rest_position, r)
    
    # Create measures with explicit measure boundaries
    part.makeMeasures(inPlace=True)
    
    # Verify and fix measure durations
    for m in part.getElementsByClass('Measure'):
        measure_duration = m.duration.quarterLength
        if abs(measure_duration - beats_per_measure) > 0.001:
            # Pad or trim measure to exactly 4 beats
            if measure_duration < beats_per_measure:
                padding = beats_per_measure - measure_duration
                r = m21_note.Rest()
                r.quarterLength = padding
                m.append(r)
    
    score.append(part)
    
    # Write to MusicXML
    try:
        score.write('musicxml', fp=output_xml)
        print(f"✓ Created {output_xml}")
    except Exception as e:
        print(f"Error: {e}")
        print("Trying with makeNotation...")
        score.write('musicxml', fp=output_xml, makeNotation=True)
        print(f"✓ Created {output_xml} (with notation fixes)")
    
    return output_xml

In [28]:
# =============================================================================
# ADVANCED: Position optimization for better fingering
# =============================================================================

def optimize_tablature_positions(jam, max_stretch=4):
    """
    Optimize string/fret positions for better playability.
    Tries to keep notes within reasonable hand positions.
    
    Args:
        jam: JAMS object with tab_note annotation
        max_stretch: Maximum fret stretch (typically 4 frets)
    
    Returns:
        Modified JAMS object with optimized positions
    """
    # Find tab annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == "tab_note":
            tab_ann = ann
            break
    
    if not tab_ann or len(tab_ann.data) < 2:
        return jam
    
    print("Optimizing tablature positions for playability...")
    
    # Re-calculate positions considering hand position
    notes = list(tab_ann.data)
    current_position = None  # (center_fret, string)
    
    for i, obs in enumerate(notes):
        pitch = obs.value['pitch']
        
        # Get all possible positions
        positions = midi_pitch_to_guitar_positions(pitch)
        
        if not positions:
            continue
        
        if current_position is None:
            # First note - choose most natural position
            string, fret = positions[0]
            current_position = fret
        else:
            # Choose position within max_stretch of current position
            valid_positions = [
                (s, f) for s, f in positions
                if abs(f - current_position) <= max_stretch
            ]
            
            if valid_positions:
                # Choose closest to current position
                string, fret = min(valid_positions, 
                                  key=lambda p: abs(p[1] - current_position))
            else:
                # No position within reach - need to shift hand position
                string, fret = positions[0]
                current_position = fret
        
        # Update the note
        obs.value['string'] = string
        obs.value['fret'] = fret
    
    print("✓ Positions optimized")
    return jam

In [29]:
def add_random_techniques_to_existing_jam(jam, technique_probability=0.5):
    '''
    Adds random techniques to EXISTING tab_note annotation
    '''
    # tech_options = ["slide", "vibrato", "hammer-on", "pull-off", "bend", "harmonic"]
    tech_options = ["harmonic"]
    # Find the EXISTING tab_note annotation
    tab_ann = None
    for ann in jam.annotations:
        if ann.namespace == 'tab_note':
            tab_ann = ann
            break
    
    if tab_ann is None:
        raise ValueError("No tab_note annotation found! Run midi_to_jams_with_tablature() first")
    
    technique_count = 0
    
    # MODIFY existing notes (don't create new ones)
    for obs in tab_ann.data:
        # Add technique to existing note
        if random.random() < technique_probability:
            tech = random.choice(tech_options)
            obs.value['techniques'] = [tech]  # ← Modify existing
            technique_count += 1
        else:
            obs.value['techniques'] = []
    
    print(f"\n✓ Modified {len(tab_ann.data)} notes")
    print(f"✓ Added {technique_count} techniques ({technique_count/len(tab_ann.data)*100:.1f}%)")
    
    return jam

In [30]:
# Step 1: Create JAMS with tablature
jam = midi_to_jams_with_tablature(midi_path)

# Step 2: Add techniques to EXISTING tab_note annotation
jam = add_random_techniques_to_existing_jam(jam, technique_probability=0.5)

# Step 3: Debug - check techniques are there
tab_ann = [a for a in jam.annotations if a.namespace == 'tab_note'][0]
print("\nNotes with techniques:")
for i, obs in enumerate(tab_ann.data[:20]):
    techs = obs.value.get('techniques', [])
    if techs:
        print(f"  Note {i}: {techs}")

# Step 4: Convert to MusicXML
jams_to_musicxml_real(jam, 'test_output', tempo_bpm=108)

Time signature from MIDI: 4/4
Extracted from MIDI - Tempo: 60.0 BPM, Time Signature: 4/4
Converted 61 notes to tablature
Pitch range: 56 - 75

✓ Modified 61 notes
✓ Added 33 techniques (54.1%)

Notes with techniques:
  Note 0: ['harmonic']
  Note 1: ['harmonic']
  Note 4: ['harmonic']
  Note 6: ['harmonic']
  Note 7: ['harmonic']
  Note 9: ['harmonic']
  Note 10: ['harmonic']
  Note 11: ['harmonic']
  Note 12: ['harmonic']
  Note 14: ['harmonic']
  Note 15: ['harmonic']
  Note 16: ['harmonic']
  Note 19: ['harmonic']
Converting 61 notes to MusicXML...
  Total duration: 60.50 beats -> 16 measures
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
Harmonic detected
  Processing 1 technique(s)
H

'test_output'